### Cell 1 — Install (run once → Kernel → Restart)

In [1]:
import subprocess, sys

def _pip(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "nvidia-nat", "nvidia-nat-core"], capture_output=True)
subprocess.run([sys.executable, "-m", "pip", "install", "git+https://github.com/NVIDIA/NeMo-Agent-Toolkit.git@v1.5.0"], capture_output=True)
print("✅ NAT v1.5.0 installed")

try:
    import pennylane as qml
    print(f"✅ PennyLane: {qml.__version__}")
except ImportError:
    _pip("pennylane"); _pip("pennylane-lightning")
    import pennylane as qml
    print(f"✅ PennyLane installed: {qml.__version__}")

import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
print("✅ Done — restart kernel now")

✅ NAT v1.5.0 installed
✅ PennyLane: 0.44.1
✅ Done — restart kernel now


### Cell 2 — Imports & config

In [2]:
import os, pickle, json, asyncio, random, warnings, math, types, gc
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from transformers import GPT2Tokenizer
from IPython.display import Markdown, display
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

try:
    import nat
    print("✓ nvidia-nat loaded")
except ImportError:
    print("⚠  nvidia-nat not installed — simulation mode")

NVIDIA_API_KEY = os.environ.get("NVIDIA_API_KEY", "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")
if NVIDIA_API_KEY and not NVIDIA_API_KEY.startswith("nvapi-PASTE"):
    print(f"✓ NVIDIA_API_KEY set  ({NVIDIA_API_KEY[:12]}...)")
else:
    print("⚠  No API key — agents will use simulation fallback")

BASE        = Path(".")
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE  = 8
MAX_LEN     = 64
# Exact split params from final.ipynb Config
TEST_SIZE   = 0.15
SPLIT_SEED  = 42
COND_LABELS = {0: "NR", 1: "TSR", 2: "SR"}

V5_BASELINE = {
    "model":         "EEG2TextTransformerV5",
    "architecture":  "Conv1D + Bi-GRU + attention EEG, MLP spectral/eye, prefix-tuned DistilGPT2",
    "tf_bleu1_pct":  29.24,  "tf_bleu4_pct": None,
    "tf_rouge1_pct": 33.92,  "tf_rougeL_pct": 30.06,
    "bertscore_f1":  None,
    "per_condition": {"NR": 30.70, "TSR": 32.78, "SR": 26.49},
    "note": "Single mean-pool EEG — no region decomp, no MoCo, no LoRA"
}
V8_BASELINE = {
    "model":         "EEG2TextTransformerV8",
    "architecture":  "6-region GRU-Transformer, MoCo Stage0, LoRA rank=8 GPT2 [10,11], SR adapter",
    "tf_bleu1_pct":  30.40,  "tf_bleu4_pct": 4.30,
    "tf_rouge1_pct": 36.01,  "tf_rougeL_pct": 30.90,
    "fg_bleu1_pct":  4.92,  "bertscore_f1": 85.46,
    "per_condition": {"NR": 30.90, "TSR": 32.93, "SR": 27.20},
    "tf_fg_ratio":  6.19,
    "note": "pool_attn collapsed → 1/256; true cross-region in self.fusion MHA"
}

print(f"Device: {DEVICE}  |  split TEST_SIZE={TEST_SIZE}  seed={SPLIT_SEED}")
print(f"V5 TF BLEU-1: {V5_BASELINE['tf_bleu1_pct']}%")
print(f"V8 TF BLEU-1: {V8_BASELINE['tf_bleu1_pct']}%  BERTScore: {V8_BASELINE['bertscore_f1']}%")

✓ nvidia-nat loaded
✓ NVIDIA_API_KEY set  (nvapi-5xnVLQ...)
Device: cuda  |  split TEST_SIZE=0.15  seed=42
V5 TF BLEU-1: 29.24%
V8 TF BLEU-1: 30.4%  BERTScore: 85.46%


### Cell 3 — Import V9 model

In [3]:
import sys, importlib
sys.path.insert(0, str(BASE))
import model1_v9
importlib.reload(model1_v9)
from model1_v9 import EEG2TextTransformerV9, REGION_NAMES, REGION_INDICES
print(f"✓ model1_v9 loaded  |  regions ({len(REGION_NAMES)}): {REGION_NAMES}")

✓ model1_v9 loaded  |  regions (6): ['left_temporal', 'left_parietal', 'left_parieto_occipital', 'central_parietal', 'right_parietal', 'right_parieto_occipital']


### Cell 4 — Load tokenizer + V9 classical checkpoint

In [4]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

model = EEG2TextTransformerV9(
    gpt_model_name="gpt2", n_heads=4, dropout=0.3,
    contrast_dim=128, region_dim=384,
).to(DEVICE)

ckpt = torch.load(BASE / "stage1_best_v9.pt", map_location="cpu")
model.load_state_dict(ckpt, strict=False); del ckpt
print("✓ Stage 1 weights loaded")

model.stage_2_setup(lora_rank=4, lora_alpha=8.0, lora_blocks=[11])
print("✓ LoRA applied")

ckpt = torch.load(BASE / "final_best_v9.pt", map_location="cpu")
model.load_state_dict(ckpt, strict=False); del ckpt
print("✓ final_best_v9.pt loaded")

model.gpt2.gradient_checkpointing_disable()
for p in model.parameters(): p.requires_grad = False
model.eval()
print(f"V9 classical  |  total={sum(p.numel() for p in model.parameters())/1e6:.1f}M  device={DEVICE}")

✓ Stage 1 weights loaded
[Stage 2] LoRA → blocks [11], rank=4
[Stage 2] Trainable: 22,656,908 / 147,096,716  (15.4%)
✓ LoRA applied
✓ final_best_v9.pt loaded
V9 classical  |  total=147.1M  device=cuda


### Cell 5 — Build QuantumFusionProjector + hybrid model

In [5]:
import pennylane as qml
import torch.nn as nn

N_QUBITS = 4
N_LAYERS = 2

_qfp_dev = qml.device("lightning.qubit", wires=N_QUBITS)

@qml.qnode(_qfp_dev, interface="torch", diff_method="adjoint")
def _eeg_vqc(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(N_QUBITS), rotation="Y")
    qml.StronglyEntanglingLayers(weights, wires=range(N_QUBITS))
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS)]


class QuantumFusionProjector(nn.Module):
    def __init__(self, H=768, n_qubits=N_QUBITS, n_layers=N_LAYERS, dropout=0.1):
        super().__init__()
        self.down  = nn.Linear(H, n_qubits)
        self.up    = nn.Linear(n_qubits, H)
        self.norm  = nn.LayerNorm(H)
        self.drop  = nn.Dropout(dropout)
        self.qlayer = qml.qnn.TorchLayer(_eeg_vqc, {"weights": (n_layers, n_qubits, 3)})
        nn.init.normal_(self.down.weight, std=0.01)
        nn.init.zeros_(self.down.bias)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        x_small = torch.tanh(self.down(x)) * torch.pi
        q_out   = self.qlayer(x_small).to(x.device)
        return self.norm(x + self.drop(self.up(q_out)))


hybrid_model = EEG2TextTransformerV9(
    gpt_model_name="gpt2", n_heads=4, dropout=0.4,  # match QML training
    contrast_dim=128, region_dim=384,
).to(DEVICE)

ckpt = torch.load(BASE / "stage1_best_v9.pt", map_location="cpu")
hybrid_model.load_state_dict(ckpt, strict=False); del ckpt
hybrid_model.stage_2_setup(lora_rank=4, lora_alpha=8.0, lora_blocks=[11])  # match QML training
hybrid_model.qfp = QuantumFusionProjector(H=768).to(DEVICE)

if (BASE / "hybrid_qml_v9_best.pt").exists():
    ckpt = torch.load(BASE / "hybrid_qml_v9_best.pt", map_location="cpu")
    hybrid_model.load_state_dict(ckpt, strict=False); del ckpt
    print("✓ hybrid_qml_v9_best.pt loaded (trained QFP weights)")
else:
    ckpt = torch.load(BASE / "final_best_v9.pt", map_location="cpu")
    hybrid_model.load_state_dict(ckpt, strict=False); del ckpt
    print("⚠  hybrid_qml_v9_best.pt not found — near-identity QFP (run QML fine-tune first)")

# sr_adapter FIRST, qfp SECOND — matches final.ipynb cell 23 training order
def _encode_eeg_hybrid(self, eeg, condition=None):
    B = eeg.size(0)
    region_tokens, region_attn_w = self.eeg_enc(eeg)
    query = self._fusion_query.expand(B, -1, -1)
    fused, fusion_attn = self.fusion(
        query, region_tokens, region_tokens,
        need_weights=True, average_attn_weights=True
    )
    fused    = self._fusion_norm(fused + query)
    fused_sq = fused.squeeze(1)
    out      = self._enc_proj_norm(fused_sq + self.enc_proj(fused_sq))
    if condition is not None:
        out = self.sr_adapter(out, condition)   # adapter FIRST
    out = self.qfp(out)                         # qfp SECOND
    self._last_region_attn_w = region_attn_w
    self._last_fusion_attn_w = fusion_attn
    return out.unsqueeze(1)

hybrid_model._encode_eeg = types.MethodType(_encode_eeg_hybrid, hybrid_model)
hybrid_model.gpt2.gradient_checkpointing_disable()
for p in hybrid_model.parameters(): p.requires_grad = False
hybrid_model.eval()
print(f"V9+QML hybrid  |  QFP={sum(p.numel() for p in hybrid_model.qfp.parameters()):,} params")

[Stage 2] LoRA → blocks [11], rank=4
[Stage 2] Trainable: 22,656,908 / 147,096,716  (15.4%)
✓ hybrid_qml_v9_best.pt loaded (trained QFP weights)
V9+QML hybrid  |  QFP=8,476 params


In [6]:
# ── Cell 5b — NoisyQuantumFusionProjector + noisy hybrid model ──────
# Loads hybrid_qml_noisy_v9_best.pt — the hardware-realistic noise-trained
# checkpoint.  Architecture must exactly match training Cell A in final.ipynb.
# q_out.to(x.device) keeps CUDA compatibility after PennyLane CPU return.
# ─────────────────────────────────────────────────────────────────────

N_QUBITS_N   = 4
N_LAYERS_N   = 2
NOISE_P_DEP  = 0.01   # DepolarizingChannel
NOISE_P_DEPH = 0.02   # PhaseDamping (T2 dephasing)
N_MC_SAMPLES = 16     # MC passes averaged at inference

_noisy_dev = qml.device("default.mixed", wires=N_QUBITS_N)

@qml.qnode(_noisy_dev, interface="torch", diff_method="best")
def _eeg_vqc_noisy(inputs, weights):
    qml.AngleEmbedding(inputs, wires=range(N_QUBITS_N), rotation="Y")
    for i in range(N_QUBITS_N):
        qml.DepolarizingChannel(NOISE_P_DEP, wires=i)
    qml.StronglyEntanglingLayers(weights, wires=range(N_QUBITS_N))
    for i in range(N_QUBITS_N):
        qml.PhaseDamping(NOISE_P_DEPH, wires=i)
    return [qml.expval(qml.PauliZ(i)) for i in range(N_QUBITS_N)]


class NoisyQuantumFusionProjector(nn.Module):
    """
    Hardware-realistic QFP (DepolarizingChannel + PhaseDamping).
    Inference: averages N_MC_SAMPLES noisy circuit passes.
    q_out.to(x.device) ensures CUDA after PennyLane CPU return.
    """
    def __init__(self, H=768, n_qubits=N_QUBITS_N, n_layers=N_LAYERS_N):
        super().__init__()
        self.n_qubits = n_qubits
        self.n_layers = n_layers
        weight_shapes = {"weights": (n_layers, n_qubits, 3)}
        self.qlayer   = qml.qnn.TorchLayer(_eeg_vqc_noisy, weight_shapes)
        self.down     = nn.Linear(H, n_qubits)
        self.up       = nn.Linear(n_qubits, H)
        self.norm     = nn.LayerNorm(H)
        self.drop     = nn.Dropout(0.1)

    def _single_pass(self, x):
        x_small = torch.tanh(self.down(x)) * torch.pi
        q_out   = self.qlayer(x_small)
        if isinstance(q_out, (list, tuple)):
            q_out = torch.stack(q_out, dim=-1)
        return q_out.to(x.device)          # PennyLane returns CPU

    def forward(self, x):
        samples = torch.stack(
            [self._single_pass(x) for _ in range(N_MC_SAMPLES)], dim=0
        )
        q_out = samples.mean(dim=0)
        return self.norm(x + self.drop(self.up(q_out)))


# ── Build noisy hybrid ────────────────────────────────────────────────
noisy_hybrid = EEG2TextTransformerV9(
    gpt_model_name="gpt2", n_heads=4, dropout=0.4,   # match training
    contrast_dim=128, region_dim=384,
).to(DEVICE)

ckpt = torch.load(BASE / "stage1_best_v9.pt", map_location="cpu")
noisy_hybrid.load_state_dict(ckpt, strict=False); del ckpt
noisy_hybrid.stage_2_setup(lora_rank=4, lora_alpha=8.0, lora_blocks=[11])
noisy_hybrid.qfp = NoisyQuantumFusionProjector(H=768).to(DEVICE)

if (BASE / "hybrid_qml_noisy_v9_best.pt").exists():
    ckpt = torch.load(BASE / "hybrid_qml_noisy_v9_best.pt", map_location="cpu")
    noisy_hybrid.load_state_dict(ckpt, strict=False); del ckpt
    print("noisy checkpoint loaded: hybrid_qml_noisy_v9_best.pt")
else:
    ckpt = torch.load(BASE / "hybrid_qml_v9_best.pt", map_location="cpu")
    noisy_hybrid.load_state_dict(ckpt, strict=False); del ckpt
    print("WARNING: noisy checkpoint not found -- using clean QML weights as proxy")

def _encode_eeg_noisy_hybrid(self, eeg, condition=None):
    B = eeg.size(0)
    region_tokens, region_attn_w = self.eeg_enc(eeg)
    query = self._fusion_query.expand(B, -1, -1)
    fused, fusion_attn = self.fusion(
        query, region_tokens, region_tokens,
        need_weights=True, average_attn_weights=True
    )
    fused    = self._fusion_norm(fused + query)
    fused_sq = fused.squeeze(1)
    out      = self._enc_proj_norm(fused_sq + self.enc_proj(fused_sq))
    if condition is not None:
        out = self.sr_adapter(out, condition)
    out = self.qfp(out)
    self._last_region_attn_w = region_attn_w
    self._last_fusion_attn_w = fusion_attn
    return out.unsqueeze(1)

noisy_hybrid._encode_eeg = types.MethodType(_encode_eeg_noisy_hybrid, noisy_hybrid)
noisy_hybrid.gpt2.gradient_checkpointing_disable()
for p in noisy_hybrid.parameters(): p.requires_grad = False
noisy_hybrid.eval()
print(f"V9+QML noisy  |  QFP={sum(p.numel() for p in noisy_hybrid.qfp.parameters()):,} params")
print(f"  Noise  : DepolarizingChannel(p={NOISE_P_DEP}) + PhaseDamping(gamma={NOISE_P_DEPH})")
print(f"  Infer  : MC average over {N_MC_SAMPLES} passes")


[Stage 2] LoRA → blocks [11], rank=4
[Stage 2] Trainable: 22,656,908 / 147,096,716  (15.4%)
noisy checkpoint loaded: hybrid_qml_noisy_v9_best.pt
V9+QML noisy  |  QFP=8,476 params
  Noise  : DepolarizingChannel(p=0.01) + PhaseDamping(gamma=0.02)
  Infer  : MC average over 16 passes


### Cell 6 — Load lean pickles, reproduce exact val split, apply saved normalisers

**Root cause of score gap:** `final.ipynb` normalises EEG (z-score via `eeg_mean.npy`/`eeg_std.npy`) and
standardises eye/spec features (sklearn `StandardScaler`) before training AND before evaluation.
The NAT notebook was loading raw un-normalised values — the model has never seen those during training.
This cell reproduces the exact pipeline:
1. Same sentence-aware val split (`TEST_SIZE=0.15, random_state=42`)
2. EEG z-score from saved `eeg_mean.npy` / `eeg_std.npy`
3. Eye + spec scaling from saved `scaler_eye.pkl` / `scaler_spec.pkl`
   (if sklearn scalers not saved, they are refit on train split exactly as `final.ipynb` does)

In [7]:
# ── 1. Load full combined pickle ──────────────────────────────
def load_pkl(path):
    with open(path, "rb") as f:
        obj = pickle.load(f)
    return obj if isinstance(obj, pd.DataFrame) else pd.DataFrame(obj)

print("Loading lean pickles...")
nr_df  = load_pkl(BASE / "NR_lean.pkl");  nr_df["condition"]  = 0
tsr_df = load_pkl(BASE / "TSR_lean.pkl"); tsr_df["condition"] = 1
sr_df  = load_pkl(BASE / "SR_lean.pkl");  sr_df["condition"]  = 2
df = pd.concat([nr_df, tsr_df, sr_df], ignore_index=True)
del nr_df, tsr_df, sr_df; gc.collect()
print(f"  Combined: {len(df):,} rows")

# ── 2. Reproduce exact sentence-aware train/val split ─────────
# final.ipynb cell 6: train_test_split(unique_sents, test_size=0.15, random_state=42)
unique_sents = df["sentence_index"].unique()
train_sents, val_sents = train_test_split(
    unique_sents, test_size=TEST_SIZE, random_state=SPLIT_SEED
)
train_df = df[df["sentence_index"].isin(train_sents)].reset_index(drop=True)
val_df   = df[df["sentence_index"].isin(val_sents)].reset_index(drop=True)
print(f"  train: {len(train_df):,} rows ({len(train_sents)} sentences)")
print(f"  val  : {len(val_df):,}  rows ({len(val_sents)} sentences)")
assert len(set(train_sents) & set(val_sents)) == 0, "Leakage!"
print("  ✓ No sentence leakage")

# ── 3. EEG z-score normalisation (MUST match training) ────────
# final.ipynb cell 7: eeg_mean.npy + eeg_std.npy saved from train_df
EYE_COLS  = ["n_fixations", "mean_fix_duration", "mean_pupilsize"]
SPEC_COLS = [
    "mean_a1_diff_mean","mean_a2_diff_mean",
    "mean_b1_diff_mean","mean_b2_diff_mean",
    "mean_g1_diff_mean","mean_g2_diff_mean",
    "mean_t1_diff_mean","mean_t2_diff_mean",
]
EYE_COLS  = [c for c in EYE_COLS  if c in df.columns]
SPEC_COLS = [c for c in SPEC_COLS if c in df.columns]

if (BASE / "eeg_mean.npy").exists() and (BASE / "eeg_std.npy").exists():
    eeg_mean = np.load(BASE / "eeg_mean.npy")   # (24,)
    eeg_std  = np.load(BASE / "eeg_std.npy")    # (24,)
    print("  ✓ Loaded eeg_mean.npy / eeg_std.npy")
else:
    # Recompute from train split — identical to final.ipynb cell 7
    print("  Computing EEG stats from train split (eeg_mean/std not found)...")
    n_ch = np.zeros(24); mean_eeg = np.zeros(24); M2_eeg = np.zeros(24)
    for eeg in tqdm(train_df["eeg"], desc="  EEG stats"):
        x = np.array(eeg, dtype=np.float64)
        for t in range(x.shape[0]):
            n_ch += 1; delta = x[t] - mean_eeg
            mean_eeg += delta / n_ch
            M2_eeg   += delta * (x[t] - mean_eeg)
    eeg_std  = np.sqrt(M2_eeg / (n_ch - 1)).astype(np.float32)
    eeg_std[eeg_std == 0] = 1.0
    eeg_mean = mean_eeg.astype(np.float32)
    np.save(BASE / "eeg_mean.npy", eeg_mean)
    np.save(BASE / "eeg_std.npy",  eeg_std)
    print("  ✓ EEG stats computed and saved")

# ── 4. Eye + spectral scalers ──────────────────────────────────
# final.ipynb cell 8: StandardScaler fitted on train_df, applied to val_df
if (BASE / "scaler_eye.pkl").exists() and (BASE / "scaler_spec.pkl").exists():
    with open(BASE / "scaler_eye.pkl",  "rb") as f: scaler_eye  = pickle.load(f)
    with open(BASE / "scaler_spec.pkl", "rb") as f: scaler_spec = pickle.load(f)
    print("  ✓ Loaded scaler_eye.pkl / scaler_spec.pkl")
else:
    print("  Fitting scalers on train split (scaler pkl not found)...")
    scaler_eye  = StandardScaler().fit(train_df[EYE_COLS].fillna(0))
    scaler_spec = StandardScaler().fit(train_df[SPEC_COLS].fillna(0))
    with open(BASE / "scaler_eye.pkl",  "wb") as f: pickle.dump(scaler_eye,  f)
    with open(BASE / "scaler_spec.pkl", "wb") as f: pickle.dump(scaler_spec, f)
    print("  ✓ Scalers fitted and saved")

# Apply scalers to val_df in-place
val_df[EYE_COLS]  = scaler_eye.transform(val_df[EYE_COLS].fillna(0))
val_df[SPEC_COLS] = scaler_spec.transform(val_df[SPEC_COLS].fillna(0))

# Apply EEG normalisation to val_df in-place
val_df["eeg"] = [
    ((np.array(x, dtype=np.float32) - eeg_mean) / eeg_std)
    for x in tqdm(val_df["eeg"], desc="  Normalising EEG")
]
gc.collect()

print(f"\n✓ val_df ready: {len(val_df):,} rows — EEG + eye + spec normalised")
print(f"  Conditions: NR={( val_df['condition']==0).sum()}  "
      f"TSR={( val_df['condition']==1).sum()}  SR={( val_df['condition']==2).sum()}")

Loading lean pickles...
  Combined: 12,952 rows
  train: 10,920 rows (345 sentences)
  val  : 2,032  rows (62 sentences)
  ✓ No sentence leakage
  ✓ Loaded eeg_mean.npy / eeg_std.npy
  ✓ Loaded scaler_eye.pkl / scaler_spec.pkl


  Normalising EEG:   0%|          | 0/2032 [00:00<?, ?it/s]


✓ val_df ready: 2,032 rows — EEG + eye + spec normalised
  Conditions: NR=639  TSR=720  SR=673


### Cell 7 — Build tensor list from normalised val_df

In [8]:
SPEC_ARR_COLS = [
    "mean_a1_diff_array","mean_a2_diff_array",
    "mean_b1_diff_array","mean_b2_diff_array",
    "mean_g1_diff_array","mean_g2_diff_array",
    "mean_t1_diff_array","mean_t2_diff_array",
]

def row_to_tensors(row):
    # EEG — already normalised in val_df, just shape-fix
    raw = np.array(row["eeg"], dtype=np.float32)
    if raw.ndim == 1:                raw = raw.reshape(-1, 24)
    elif raw.shape[0] < raw.shape[1]: raw = raw.T
    T, C = raw.shape
    if T < 256: raw = np.pad(raw, ((0, 256-T), (0, 0)))
    else:        raw = raw[:256, :]
    if C < 24:  raw = np.pad(raw, ((0, 0), (0, 24-C)))
    else:        raw = raw[:, :24]

    # Sentence-level spectral — already scaled in val_df
    spec = np.array(row[SPEC_COLS].values, dtype=np.float32)

    # Word-level spectral → (50, 8)  [per-band individual pad → stack]
    # Matches MultimodalEEGDataset exactly: each band padded to 50, then np.stack(..., axis=1)
    word_arrays = []
    for c in SPEC_ARR_COLS:
        v = row.get(c, None)
        if v is not None:
            a = np.array(v).flatten()
            a = a[:50] if len(a) >= 50 else np.pad(a, (0, 50-len(a)))
        else:
            a = np.zeros(50)
        word_arrays.append(a.astype(np.float32))
    specw_2d = np.stack(word_arrays, axis=1)   # (50, 8)

    # Eye-tracking — already scaled in val_df
    eye  = np.array(row[EYE_COLS].values, dtype=np.float32)
    cond = int(row["condition"])
    sent = str(row["sentence"])
    subj = str(row.get("subject_id", "?"))
    return raw, spec, specw_2d, eye, cond, sent, subj

print("Building tensor list from normalised val_df...")
all_eeg, all_spec, all_specw, all_eye = [], [], [], []
all_cond, all_sentences, all_subjects = [], [], []

for _, row in tqdm(val_df.iterrows(), total=len(val_df)):
    raw, spec, specw_2d, eye, cond, sent, subj = row_to_tensors(row)
    all_eeg.append(raw)
    all_spec.append(spec)
    all_specw.append(specw_2d)
    all_eye.append(eye)
    all_cond.append(cond)
    all_sentences.append(sent)
    all_subjects.append(subj)

n_total   = len(all_sentences)
n_batches = math.ceil(n_total / BATCH_SIZE)
print(f"\nTensors ready: {n_total:,} rows  ({n_batches} batches)")
print(f"  eeg:   {all_eeg[0].shape}   (256,24) normalised")
print(f"  specw: {all_specw[0].shape} (50,8)   per-band stacked")
print(f"  eye:   {all_eye[0].shape}    (3,)     scaled")
print(f"  NR={all_cond.count(0):,}  TSR={all_cond.count(1):,}  SR={all_cond.count(2):,}")

Building tensor list from normalised val_df...


  0%|          | 0/2032 [00:00<?, ?it/s]


Tensors ready: 2,032 rows  (254 batches)
  eeg:   (256, 24)   (256,24) normalised
  specw: (50, 8) (50,8)   per-band stacked
  eye:   (3,)    (3,)     scaled
  NR=639  TSR=720  SR=673


### Cell 8 — Diagnose HTP attention (V9)

In [9]:
print("Attention weight norms per region (V9 HTP):")
for name in REGION_NAMES:
    enc  = model.eeg_enc.region_encoders[name]
    mods = {}
    for attr in ["pool_attn", "local_attn", "seg_attn"]:
        if hasattr(enc, attr):
            mods[attr] = getattr(enc, attr).weight.norm().item()
    parts     = [f"{k}={v:.6f}" for k, v in mods.items()]
    collapsed = any(v < 0.01 for v in mods.values())
    flag      = "  ⚠ COLLAPSED" if collapsed else "  ✓ selective"
    print(f"  {name:28s}  {' | '.join(parts)}{flag}")
print()
print("V8 baseline: pool_attn uniform 1/256 = 0.003906")
print("V9 HTP goal: local_attn + seg_attn learn meaningful temporal peaks")

Attention weight norms per region (V9 HTP):
  left_temporal                   ✓ selective
  left_parietal                   ✓ selective
  left_parieto_occipital          ✓ selective
  central_parietal                ✓ selective
  right_parietal                  ✓ selective
  right_parieto_occipital         ✓ selective

V8 baseline: pool_attn uniform 1/256 = 0.003906
V9 HTP goal: local_attn + seg_attn learn meaningful temporal peaks


### Cell 9 — Patch classical V9 to capture fusion attention

In [10]:
def _encode_eeg_classical_patched(self, eeg, condition=None):
    B = eeg.size(0)
    region_tokens, region_attn_w = self.eeg_enc(eeg)
    query = self._fusion_query.expand(B, -1, -1)
    fused, fusion_attn = self.fusion(
        query, region_tokens, region_tokens,
        need_weights=True, average_attn_weights=True
    )
    fused    = self._fusion_norm(fused + query)
    fused_sq = fused.squeeze(1)
    out      = self._enc_proj_norm(fused_sq + self.enc_proj(fused_sq))
    if condition is not None:
        out = self.sr_adapter(out, condition)
    self._last_region_attn_w = region_attn_w
    self._last_fusion_attn_w = fusion_attn
    return out.unsqueeze(1)

model._encode_eeg = types.MethodType(_encode_eeg_classical_patched, model)
print("✓ V9 classical patched — fusion_attn captured")
print("✓ Hybrid V9+QML already captures fusion_attn (Cell 5)")

✓ V9 classical patched — fusion_attn captured
✓ Hybrid V9+QML already captures fusion_attn (Cell 5)


### Cell 10 — Batched inference: TF + FG + attention for both models

In [11]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer as rs_module

eos      = tokenizer.eos_token_id
smoother = SmoothingFunction().method1
rouge_sc = rs_module.RougeScorer(["rouge1", "rougeL"], use_stemmer=False)

def decode_clean(ids_tensor, ref_len=None):
    """Decode + strip EOS + cap at ref length + remove 3-repeat stutters."""
    out = []
    for i, row in enumerate(ids_tensor):
        lst = row.tolist()
        if eos in lst: lst = lst[:lst.index(eos)]
        if ref_len is not None: lst = lst[:ref_len[i]]
        tokens = tokenizer.decode(lst, skip_special_tokens=True).split()
        clean = []
        for t in tokens:
            if len(clean) >= 3 and all(x.lower() == t.lower() for x in clean[-3:]):
                break
            clean.append(t)
        out.append(" ".join(clean))
    return out

results = {
    "classical": {
        "tf_preds": [], "fg_preds": [],
        "temporal":    {n: 0.0 for n in REGION_NAMES},
        "cross":       {n: 0.0 for n in REGION_NAMES},
        "cond_cross":  {c: {n: 0.0 for n in REGION_NAMES} for c in ["NR","TSR","SR"]},
        "cond_counts": {"NR": 0, "TSR": 0, "SR": 0},
    },
    "hybrid": {
        "tf_preds": [], "fg_preds": [],
        "temporal":    {n: 0.0 for n in REGION_NAMES},
        "cross":       {n: 0.0 for n in REGION_NAMES},
        "cond_cross":  {c: {n: 0.0 for n in REGION_NAMES} for c in ["NR","TSR","SR"]},
        "cond_counts": {"NR": 0, "TSR": 0, "SR": 0},
    },
    "noisy_hybrid": {
        "tf_preds": [], "fg_preds": [],
        "temporal":    {n: 0.0 for n in REGION_NAMES},
        "cross":       {n: 0.0 for n in REGION_NAMES},
        "cond_cross":  {c: {n: 0.0 for n in REGION_NAMES} for c in ["NR","TSR","SR"]},
        "cond_counts": {"NR": 0, "TSR": 0, "SR": 0},
    },
}

print(f"Running inference on normalised val set: {n_total:,} rows ({n_batches} batches)")

for b in tqdm(range(n_batches)):
    s = b * BATCH_SIZE
    e = min(s + BATCH_SIZE, n_total)
    B = e - s

    eeg_b   = torch.tensor(np.stack(all_eeg[s:e]),   dtype=torch.float32).to(DEVICE)
    spec_b  = torch.tensor(np.stack(all_spec[s:e]),  dtype=torch.float32).to(DEVICE)
    specw_b = torch.tensor(np.stack(all_specw[s:e]), dtype=torch.float32).to(DEVICE)
    eye_b   = torch.tensor(np.stack(all_eye[s:e]),   dtype=torch.float32).to(DEVICE)
    cond_b  = torch.tensor(all_cond[s:e],            dtype=torch.long).to(DEVICE)

    enc    = tokenizer(
        all_sentences[s:e], return_tensors="pt",
        padding=True, truncation=True, max_length=MAX_LEN
    )
    ids_b    = enc["input_ids"].to(DEVICE)
    ref_lens = [
        ids_b[i].tolist().index(eos) if eos in ids_b[i].tolist() else MAX_LEN
        for i in range(B)
    ]

    with torch.no_grad():
        for mdl_key, mdl in [("classical", model), ("hybrid", hybrid_model), ("noisy_hybrid", noisy_hybrid)]:
            r = results[mdl_key]

            tf_logits = mdl(eeg_b, eye_b, spec_b, specw_b, cond_b, ids_b)
            tf_ids    = tf_logits.argmax(dim=-1)
            r["tf_preds"].extend(decode_clean(tf_ids, ref_lens))

            wt = mdl._last_region_attn_w
            if isinstance(wt, (list, tuple)):
                for i, name in enumerate(REGION_NAMES):
                    w = wt[i]
                    if isinstance(w, (list, tuple)): w = w[0]
                    r["temporal"][name] += float(w.mean().item()) * B

            if hasattr(mdl, "_last_fusion_attn_w"):
                fw = mdl._last_fusion_attn_w.squeeze(1)
                for i, name in enumerate(REGION_NAMES):
                    r["cross"][name] += float(fw[:, i].mean().item()) * B
                for j in range(B):
                    cname = COND_LABELS.get(all_cond[s+j], "?")
                    if cname in r["cond_cross"]:
                        r["cond_counts"][cname] += 1
                        for i, name in enumerate(REGION_NAMES):
                            r["cond_cross"][cname][name] += float(fw[j, i].item())

            fg_ids = mdl.generate_text(
                eeg_b, eye_b, spec_b, specw_b, cond_b,
                tokenizer, max_len=MAX_LEN, eeg_alpha=0.0,
                num_beams=1, do_sample=False,
            )
            r["fg_preds"].extend(decode_clean(fg_ids))

for mdl_key in ["classical", "hybrid", "noisy_hybrid"]:
    r = results[mdl_key]
    r["temporal_norm"]   = {k: round(v/n_total, 6) for k, v in r["temporal"].items()}
    r["cross_norm"]      = {k: round(v/n_total, 6) for k, v in r["cross"].items()}
    r["cond_cross_norm"] = {
        cname: {name: round(r["cond_cross"][cname][name]/max(r["cond_counts"][cname],1), 6)
                for name in REGION_NAMES}
        for cname in ["NR","TSR","SR"]
    }
    r["dominant_cross"] = max(r["cross_norm"], key=r["cross_norm"].get)

print("\n── V9 classical dominant:   ", results["classical"]["dominant_cross"])
print("── V9+QML clean dominant:   ", results["hybrid"]["dominant_cross"])
print("── V9+QML noisy dominant:   ", results["noisy_hybrid"]["dominant_cross"])

Running inference on normalised val set: 2,032 rows (254 batches)


  0%|          | 0/254 [00:00<?, ?it/s]


── V9 classical dominant:    left_parieto_occipital
── V9+QML clean dominant:    left_parieto_occipital
── V9+QML noisy dominant:    left_parieto_occipital


### Cell 11 — Compute BLEU / ROUGE metrics

In [12]:
def corpus_metrics(refs, hyps, label=""):
    b1, b4, r1, rl = [], [], [], []
    for ref, hyp in zip(refs, hyps):
        rt, ht = ref.lower().split(), hyp.lower().split()
        if not rt or not ht: continue
        b1.append(sentence_bleu([rt], ht, weights=(1,0,0,0), smoothing_function=smoother))
        b4.append(sentence_bleu([rt], ht, weights=(.25,.25,.25,.25), smoothing_function=smoother))
        rg = rouge_sc.score(ref, hyp)
        r1.append(rg["rouge1"].fmeasure)
        rl.append(rg["rougeL"].fmeasure)
    n = len(b1)
    if n == 0: return {"n":0,"bleu1":0,"bleu4":0,"rouge1":0,"rougeL":0}
    m = {"n":n,
         "bleu1":  round(sum(b1)/n*100,2),
         "bleu4":  round(sum(b4)/n*100,2),
         "rouge1": round(sum(r1)/n*100,2),
         "rougeL": round(sum(rl)/n*100,2)}
    if label:
        print(f"  {label}: BLEU-1={m['bleu1']}%  BLEU-4={m['bleu4']}%  "
              f"ROUGE-1={m['rouge1']}%  ROUGE-L={m['rougeL']}%")
    return m

print("=" * 65)
print(f"Val split metrics  (n={n_total:,}, same split as final.ipynb)")
print("=" * 65)

for mdl_key in ["classical", "hybrid", "noisy_hybrid"]:
    r     = results[mdl_key]
    label = {"classical": "V9 classical",
             "hybrid":       "V9+QML clean",
             "noisy_hybrid": "V9+QML noisy"}.get(mdl_key, mdl_key)
    print(f"\n── {label} ──")
    r["tf_metrics"] = corpus_metrics(all_sentences, r["tf_preds"], "TF")
    r["fg_metrics"] = corpus_metrics(all_sentences, r["fg_preds"], "FG")
    tf_fg_gap   = round(r["tf_metrics"]["bleu1"] - r["fg_metrics"]["bleu1"], 2)
    tf_fg_ratio = round(r["tf_metrics"]["bleu1"] / max(r["fg_metrics"]["bleu1"], 0.01), 2)
    r["tf_fg_gap"]   = tf_fg_gap
    r["tf_fg_ratio"] = tf_fg_ratio
    print(f"  TF–FG gap: {tf_fg_gap:+.2f}pp  ratio: {tf_fg_ratio:.2f}×  (V8:6.19x)")

    print("  Per-condition TF BLEU-1:")
    r["per_cond"] = {}
    for cid, cname in COND_LABELS.items():
        idx = [i for i, c in enumerate(all_cond) if c == cid]
        if not idx: continue
        m     = corpus_metrics([all_sentences[i] for i in idx],
                               [r["tf_preds"][i]  for i in idx])
        r["per_cond"][cname] = m
        v8v   = V8_BASELINE["per_condition"].get(cname, 0)
        delta = round(m["bleu1"] - v8v, 2)
        print(f"    {cname}: {m['bleu1']}%  (V8={v8v}%  Δ={delta:+.2f}pp)  n={m['n']:,}")

print()
print(f"  V5  TF BLEU-1: {V5_BASELINE['tf_bleu1_pct']}%  ROUGE-1: {V5_BASELINE['tf_rouge1_pct']}%")
print(f"  V8  TF BLEU-1: {V8_BASELINE['tf_bleu1_pct']}%  ROUGE-1: {V8_BASELINE['tf_rouge1_pct']}%  BERTScore: {V8_BASELINE['bertscore_f1']}%")

Val split metrics  (n=2,032, same split as final.ipynb)

── V9 classical ──
  TF: BLEU-1=30.97%  BLEU-4=4.45%  ROUGE-1=36.07%  ROUGE-L=30.76%
  FG: BLEU-1=6.9%  BLEU-4=0.68%  ROUGE-1=10.35%  ROUGE-L=9.14%
  TF–FG gap: +24.07pp  ratio: 4.49×  (V8:6.19x)
  Per-condition TF BLEU-1:
    NR: 31.53%  (V8=30.9%  Δ=+0.63pp)  n=639
    TSR: 33.77%  (V8=32.93%  Δ=+0.84pp)  n=720
    SR: 27.46%  (V8=27.2%  Δ=+0.26pp)  n=673

── V9+QML clean ──
  TF: BLEU-1=30.95%  BLEU-4=4.46%  ROUGE-1=36.04%  ROUGE-L=30.77%
  FG: BLEU-1=6.88%  BLEU-4=0.67%  ROUGE-1=10.33%  ROUGE-L=9.14%
  TF–FG gap: +24.07pp  ratio: 4.50×  (V8:6.19x)
  Per-condition TF BLEU-1:
    NR: 31.43%  (V8=30.9%  Δ=+0.53pp)  n=639
    TSR: 33.87%  (V8=32.93%  Δ=+0.94pp)  n=720
    SR: 27.37%  (V8=27.2%  Δ=+0.17pp)  n=673

── V9+QML noisy ──
  TF: BLEU-1=30.95%  BLEU-4=4.46%  ROUGE-1=36.05%  ROUGE-L=30.76%
  FG: BLEU-1=6.81%  BLEU-4=0.65%  ROUGE-1=10.24%  ROUGE-L=9.14%
  TF–FG gap: +24.14pp  ratio: 4.54×  (V8:6.19x)
  Per-condition TF BLEU

### Cell 12 — Qualitative samples (one per condition)

In [13]:
print("── Qualitative samples ──────────────────────────────────────")
qual_samples = []
seen = set()
for i, (sent, cond_id) in enumerate(zip(all_sentences, all_cond)):
    cname = COND_LABELS.get(cond_id, "?")
    if cname in seen: continue
    seen.add(cname)
    entry = {
        "condition":   cname,
        "target":      sent,
        "v9_tf_pred":  results["classical"]["tf_preds"][i],
        "v9_fg_pred":  results["classical"]["fg_preds"][i],
        "qml_tf_pred": results["hybrid"]["tf_preds"][i],
        "qml_fg_pred": results["hybrid"]["fg_preds"][i],
    }
    qual_samples.append(entry)
    print(f"\n[{cname}]")
    print(f"  TARGET      : {sent}")
    print(f"  V9 TF       : {entry['v9_tf_pred']}")
    print(f"  V9 FG       : {entry['v9_fg_pred']}")
    print(f"  V9+QML TF   : {entry['qml_tf_pred']}")
    print(f"  V9+QML FG   : {entry['qml_fg_pred']}")
    if len(seen) == 3: break

── Qualitative samples ──────────────────────────────────────

[NR]
  TARGET      : Henry Ford, with his son Edsel, founded the Ford Foundation in 1936 as a local philanthropic organization with a broad charter to promote human welfare.
  V9 TF       : Ford, a his wife,,, was Ford Ford Motor, 18. a philanthrop philanthropic organization. the mission focus of promote the rights. Ford
  V9 FG       : The first of the two sons of the late President George W. Bush, Bush was born in New York City on September 11, 1963. He was the son of a former president and a former vice president. He was married to actress Mary Ann Bush, who was a member of the family of the late President George W.
  V9+QML TF   : Ford, a his wife,ith, was Ford Ford Motor, 18. a philanthrop philanthropic organization. the mission focus of support the rights. Ford
  V9+QML FG   : The first of the two sons of the late President George W. Bush, Bush was born in New York City on September 11, 1963. He was the son of a forme

### Cell 13 — Assemble agent_stats (V5 / V8 / V9 / QML)

In [14]:
c    = results["classical"]
h    = results["hybrid"]
nh   = results["noisy_hybrid"]
tf_c = c["tf_metrics"]
fg_c = c["fg_metrics"]
tf_h = h["tf_metrics"]
fg_h = h["fg_metrics"]
tf_nh = nh["tf_metrics"]
fg_nh = nh["fg_metrics"]

agent_stats = {
    "experiment": {
        "model_v9_classical": "EEG2TextTransformerV9 (HTP + LoRA rank=8)",
        "model_v9_qml":         "EEG2TextTransformerV9 + QuantumFusionProjector clean (4 qubits, 2 layers)",
        "model_v9_qml_noisy":   "EEG2TextTransformerV9 + NoisyQuantumFusionProjector (DepolarizingChannel+PhaseDamping)",
        "dataset":            "ZuCo — sentence-aware val split (TEST_SIZE=0.15, seed=42)",
        "n_val_rows":         n_total,
        "n_per_condition":    {COND_LABELS[c]: all_cond.count(c) for c in [0,1,2]},
        "subjects":           sorted(set(all_subjects)),
        "normalisation":      "EEG z-score (eeg_mean/std from train), eye+spec StandardScaler",
        "qml_circuit":        f"{N_QUBITS} qubits, {N_LAYERS} StronglyEntanglingLayers, AngleEmbedding",
        "qfp_params":         sum(p.numel() for p in hybrid_model.qfp.parameters()),
        "qml_finetune":       "5 epochs, batch=4, accum=2, QML_LR=3e-4, rest=5e-5, patience=3",
    },
    "live_metrics": {
        "n":                       n_total,
        "v9_tf_bleu1_pct":         tf_c["bleu1"],
        "v9_tf_bleu4_pct":         tf_c["bleu4"],
        "v9_tf_rouge1_pct":        tf_c["rouge1"],
        "v9_tf_rougeL_pct":        tf_c["rougeL"],
        "v9_fg_bleu1_pct":         fg_c["bleu1"],
        "v9_tf_fg_gap_pp":         c["tf_fg_gap"],
        "v9_tf_fg_ratio":          c["tf_fg_ratio"],
        "v9_per_cond_bleu1":       {k: v["bleu1"] for k, v in c["per_cond"].items()},
        "qml_tf_bleu1_pct":        tf_h["bleu1"],
        "qml_tf_bleu4_pct":        tf_h["bleu4"],
        "qml_tf_rouge1_pct":       tf_h["rouge1"],
        "qml_tf_rougeL_pct":       tf_h["rougeL"],
        "qml_fg_bleu1_pct":        fg_h["bleu1"],
        "qml_tf_fg_gap_pp":        h["tf_fg_gap"],
        "qml_tf_fg_ratio":         h["tf_fg_ratio"],
        "qml_per_cond_bleu1":      {k: v["bleu1"] for k, v in h["per_cond"].items()},
        "delta_v9_vs_v8_bleu1":    round(tf_c["bleu1"]  - V8_BASELINE["tf_bleu1_pct"],  2),
        "delta_v9_vs_v8_rouge1":   round(tf_c["rouge1"] - V8_BASELINE["tf_rouge1_pct"], 2),
        "delta_v9_vs_v5_bleu1":    round(tf_c["bleu1"]  - V5_BASELINE["tf_bleu1_pct"],  2),
        "delta_qml_vs_v9_bleu1":   round(tf_h["bleu1"]  - tf_c["bleu1"],  2),
        "delta_qml_vs_v9_rouge1":  round(tf_h["rouge1"] - tf_c["rouge1"], 2),
        "delta_qml_vs_v8_bleu1":        round(tf_h["bleu1"]  - V8_BASELINE["tf_bleu1_pct"],  2),
        "noisy_qml_tf_bleu1_pct":       tf_nh["bleu1"],
        "noisy_qml_tf_bleu4_pct":       tf_nh["bleu4"],
        "noisy_qml_tf_rouge1_pct":      tf_nh["rouge1"],
        "noisy_qml_tf_rougeL_pct":      tf_nh["rougeL"],
        "noisy_qml_fg_bleu1_pct":       fg_nh["bleu1"],
        "noisy_qml_tf_fg_gap_pp":       nh["tf_fg_gap"],
        "noisy_qml_tf_fg_ratio":        nh["tf_fg_ratio"],
        "noisy_qml_per_cond_bleu1":     {k: v["bleu1"] for k, v in nh["per_cond"].items()},
        "delta_noisy_vs_clean_bleu1":   round(tf_nh["bleu1"]  - tf_h["bleu1"],  2),
        "delta_noisy_vs_clean_rouge1":  round(tf_nh["rouge1"] - tf_h["rouge1"], 2),
        "delta_noisy_vs_v8_bleu1":      round(tf_nh["bleu1"]  - V8_BASELINE["tf_bleu1_pct"],  2),
    },
    "baselines": {"v5": V5_BASELINE, "v8": V8_BASELINE},
    "attention_analysis": {
        "v9_classical": {
            "temporal_pooling": {
                "values":    c["temporal_norm"],
                "diagnosis": "V9 HTP: local_attn + seg_attn replace V8's collapsed pool_attn",
            },
            "cross_region_fusion": {
                "values":        c["cross_norm"],
                "dominant":      c["dominant_cross"],
                "per_condition": c["cond_cross_norm"],
            },
        },
        "v9_qml_hybrid": {
            "temporal_pooling": {
                "values":    h["temporal_norm"],
                "diagnosis": "Same HTP; QFP residual acts post-adapter",
            },
            "cross_region_fusion": {
                "values":        h["cross_norm"],
                "dominant":      h["dominant_cross"],
                "per_condition": h["cond_cross_norm"],
            },
            "qfp_role": "VQC residual post-sr_adapter: down H→4, AngleEmbed+StronglyEntangle, up 4→H, LN",
        },
        "v9_qml_noisy_hybrid": {
            "temporal_pooling": {
                "values":    nh["temporal_norm"],
                "diagnosis": "Same HTP; NoisyQFP residual post-adapter (MC-averaged inference)",
            },
            "cross_region_fusion": {
                "values":        nh["cross_norm"],
                "dominant":      nh["dominant_cross"],
                "per_condition": nh["cond_cross_norm"],
            },
            "qfp_role": (
                "VQC residual with DepolarizingChannel(p=0.01) + PhaseDamping(0.02); "
                f"inference averages {N_MC_SAMPLES} noisy passes"
            ),
        },
    },
    "qualitative_samples": qual_samples,
}

lm = agent_stats["live_metrics"]
print("agent_stats assembled.")
print(f"  V5        TF BLEU-1 : {V5_BASELINE['tf_bleu1_pct']}%")
print(f"  V8        TF BLEU-1 : {V8_BASELINE['tf_bleu1_pct']}%")
print(f"  V9        TF BLEU-1 : {lm['v9_tf_bleu1_pct']}%   (D vs V8: {lm['delta_v9_vs_v8_bleu1']:+.2f}pp)")
print(f"  QML clean TF BLEU-1 : {lm['qml_tf_bleu1_pct']}%   (D vs V9: {lm['delta_qml_vs_v9_bleu1']:+.2f}pp)")
print(f"  QML noisy TF BLEU-1 : {lm['noisy_qml_tf_bleu1_pct']}%   (D vs clean: {lm['delta_noisy_vs_clean_bleu1']:+.2f}pp)")


agent_stats assembled.
  V5        TF BLEU-1 : 29.24%
  V8        TF BLEU-1 : 30.4%
  V9        TF BLEU-1 : 30.97%   (D vs V8: +0.57pp)
  QML clean TF BLEU-1 : 30.95%   (D vs V9: -0.02pp)
  QML noisy TF BLEU-1 : 30.95%   (D vs clean: +0.00pp)


### Cell 14b — Install NeMo Guardrails + benchmark deps

In [15]:
# Cell 14b — Install NeMo Guardrails + benchmark deps (run once)
import subprocess, sys

def _pip(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    import nemoguardrails
    print(f"✅ nemoguardrails already installed: {nemoguardrails.__version__}")
except ImportError:
    print("Installing nemoguardrails...")
    _pip("nemoguardrails>=0.10.0")
    import nemoguardrails
    print(f"✅ nemoguardrails {nemoguardrails.__version__}")

# openai is needed for AsyncOpenAI streaming
try:
    import openai
    print(f"✅ openai {openai.__version__}")
except ImportError:
    _pip("openai>=1.0.0")
    print("✅ openai installed")

print("All deps ready.")


✅ nemoguardrails already installed: 0.21.0
✅ openai 2.30.0
All deps ready.


### Cell 14 — Agent system prompts

In [32]:
# Cell 14 — Agent system prompts (domain-specific, upgraded from original)
# These live in eeg_product/nat_agents_guardrailed.py  — edit them there to change agent behaviour.
# This cell imports and prints them so you can review before running Cell 16.
import importlib, sys
from pathlib import Path

EEG_PRODUCT_DIR = Path(".") / "eeg_product"
if str(EEG_PRODUCT_DIR) not in sys.path:
    sys.path.insert(0, str(EEG_PRODUCT_DIR))

import nat_agents_guardrailed as nag
importlib.reload(nag)

for name, prompt in [
    ("SCIENTIST_SYSTEM",         nag.SCIENTIST_SYSTEM),
    ("CRITIC_SYSTEM",            nag.CRITIC_SYSTEM),
    ("QML_SYSTEM (Synthesiser)", nag.QML_SYSTEM),
]:
    print(f"{'='*68}\n  {name}\n{'='*68}")
    print(prompt)
    print()

print("── What changed vs original nat notebook ───────────────────────────")
print("  Scientist : [ROLE] tag + Out of scope + corrected 10 epochs, LR=1e-6")
print("  Critic    : corrected V8 baselines ROUGE-1=35.78%, BERTScore=85.46%")
print("  Explainer : replaced by QML Synthesiser — QFP circuit focus + VQC honesty")


✅ NeMo Guardrails loaded
  SCIENTIST_SYSTEM
[ROLE: scientist]
You are a neuroscience and NLP researcher analysing the four-model EEG-to-text progression on ZuCo.

Architecture evolution:
  V5  → Conv1D + Bi-GRU + single mean-pool EEG vector, prefix-tuned DistilGPT2
  V8  → 6 parallel GRU-Transformer RegionEncoders, MoCo Stage0, LoRA rank=8 GPT2 [10,11], SR adapter
         pool_attn collapsed → uniform 1/256 (mean-pooling in disguise)
         True cross-region signal lives in self.fusion MHA (was discarded as `_` in V8)
  V9  → V8 + HierarchicalTemporalPooling (HTP): local_attn + seg_attn per region
         LoRA rank=4, lora_alpha=16, block=[11] only; dropout=0.3 for eval
  QML clean  → V9 + QuantumFusionProjector AFTER sr_adapter (noiseless, lightning.qubit):
               down H→4, AngleEmbedding, 2 StronglyEntanglingLayers (4 qubits), up 4→H, LN residual
               ~8,476 QML params; 10-epoch fine-tune (QML_LR=3e-4, rest=1e-6, CosineAnnealingLR)
               dropout=0.4; lo

### Cell 14c — LLM caller + guardrail flow

In [33]:
# Cell 14c — LLM caller + guardrail flow (read-only explainer, no execution needed)
# Replaces the original "Cell 15 — LLM caller + three-agent pipeline" section.
# The actual call function is in eeg_product/nat_agents_guardrailed.py -> call_nim_guardrailed()

print("── call_nim_guardrailed() vs original call_nim() ───────────────────")
print("""
ORIGINAL call_nim()  (nat_eeg_agents_v9_correct.ipynb cell 29):
  - Direct AsyncOpenAI POST to integrate.api.nvidia.com
  - No streaming  →  no TTFT measurement
  - No guardrails  →  any output accepted
  - Returns: str

NEW call_nim_guardrailed()  (eeg_product/nat_agents_guardrailed.py):
  - Path A (NeMo installed): rails.generate_async()
      Input rail  : check eeg domain intent + validate agent_stats payload
      Dialog rail : per-agent scope guard (scientist / critic / qml_synthesiser)
      Output rail : metric bounds check (BLEU 20-55%, BERTScore 78-96%)
                    domain relevance check (≥3 EEG terms required)
  - Path B (fallback): streaming AsyncOpenAI + Python-side guards
      Measures TTFT (time-to-first-token via perf_counter)
      Measures tokens/s
  - Returns: (response_text, timing_dict)

ENDPOINT:
  Default  →  https://integrate.api.nvidia.com/v1   (your existing NVIDIA key)
  Override →  export NIM_BASE_URL=http://localhost:8000/v1   (self-hosted NIM on brev)
""")

from pathlib import Path
config_dir = Path(".") / "eeg_product" / "guardrails_config"
if config_dir.exists():
    print("── Guardrails config files loaded from eeg_product/guardrails_config/ ──")
    for f in sorted(config_dir.iterdir()):
        size = f.stat().st_size
        print(f"  {f.name:<30} {size:>6} bytes")
else:
    print("  WARNING: eeg_product/guardrails_config/ not found — check folder path")


── call_nim_guardrailed() vs original call_nim() ───────────────────

ORIGINAL call_nim()  (nat_eeg_agents_v9_correct.ipynb cell 29):
  - Direct AsyncOpenAI POST to integrate.api.nvidia.com
  - No streaming  →  no TTFT measurement
  - No guardrails  →  any output accepted
  - Returns: str

NEW call_nim_guardrailed()  (eeg_product/nat_agents_guardrailed.py):
  - Path A (NeMo installed): rails.generate_async()
      Input rail  : check eeg domain intent + validate agent_stats payload
      Dialog rail : per-agent scope guard (scientist / critic / qml_synthesiser)
      Output rail : metric bounds check (BLEU 20-55%, BERTScore 78-96%)
                    domain relevance check (≥3 EEG terms required)
  - Path B (fallback): streaming AsyncOpenAI + Python-side guards
      Measures TTFT (time-to-first-token via perf_counter)
      Measures tokens/s
  - Returns: (response_text, timing_dict)

ENDPOINT:
  Default  →  https://integrate.api.nvidia.com/v1   (your existing NVIDIA key)
  Override →

### Cell 15 — Load guardrailed pipeline module

In [34]:
# Cell 15 — Load guardrailed agent module + benchmark harness
import importlib, sys
from pathlib import Path

# Add eeg_product dir to path  (adjust if needed)
EEG_PRODUCT_DIR = Path(".") / "eeg_product"
if str(EEG_PRODUCT_DIR) not in sys.path:
    sys.path.insert(0, str(EEG_PRODUCT_DIR))

import nat_agents_guardrailed as nag
importlib.reload(nag)

print(f"✅ nat_agents_guardrailed loaded")
print(f"   NIM endpoint : {nag.NIM_BASE_URL}")
print(f"   Model        : {nag.NIM_MODEL}")
print(f"   API key set  : {bool(nag.NVIDIA_API_KEY and not nag.NVIDIA_API_KEY.startswith('nvapi-PASTE'))}")

# ── Optional: override endpoint to self-hosted NIM ────────────────────────────
# Uncomment and set NIM_BASE_URL env var before running if you have a local NIM:
#   import os
#   os.environ["NIM_BASE_URL"] = "http://localhost:8000/v1"
#   os.environ["NIM_MODEL"]    = "meta/llama-3.1-70b-instruct"
#   importlib.reload(nag)

# Print agent role system prompts (first 120 chars each)
for name, prompt in [("SCIENTIST", nag.SCIENTIST_SYSTEM),
                      ("CRITIC", nag.CRITIC_SYSTEM),
                      ("QML_SYNTHESISER", nag.QML_SYSTEM)]:
    print(f"\n  {name}: {prompt[:120].strip()}...")


✅ NeMo Guardrails loaded
✅ nat_agents_guardrailed loaded
   NIM endpoint : https://integrate.api.nvidia.com/v1
   Model        : meta/llama-3.1-8b-instruct
   API key set  : True

  SCIENTIST: [ROLE: scientist]
You are a neuroscience and NLP researcher analysing the four-model EEG-to-text progression on ZuCo.

A...

  CRITIC: [ROLE: critic]
You are a senior reviewer at NeurIPS / IEEE TNSRE evaluating an EEG-to-text decoding paper.

Submission e...

  QML_SYNTHESISER: [ROLE: qml_synthesiser]
You are a quantum-ML researcher and science communicator writing for a final-year engineering st...


### Cell 16 — Run guardrailed 3-agent pipeline

In [35]:
# Cell 16 — Run guardrailed 3-agent pipeline with benchmarking
# Replaces original run_pipeline() / cell 29

results_agents = await nag.run_guardrailed_pipeline(agent_stats)

# Quick guardrail audit print
print("\n── Guardrail audit ──────────────────────────────────────────────")
for rec in results_agents["guardrail_audit"]:
    status = "✅ PASS" if rec["guardrail_pass"] else f"⛔ FIRED: {rec['guardrail_fired']}"
    print(f"  {rec['agent']:<18} {status:<35} "
          f"latency={rec['total_ms']}ms  TTFT={rec['ttft_ms']}ms  tok/s={rec['tokens_per_sec']}")

ps = results_agents["pipeline_summary"]
print(f"\n  Total pipeline : {ps['total_pipeline_ms']}ms")
print(f"  Pass rate      : {ps['guardrail_pass_rate_pct']}%")
print(f"  Endpoint       : {ps['endpoint']}")
print(f"  Rails active   : {ps['rails_active']}")


✅ Guardrails loaded (Colang 1.0) from /home/deeptanshu/Desktop/project1/eeg_product/guardrails_config
  EEG2TextTransformerV9+QML — Guardrailed Three-Agent Pipeline
  Val n=2,032  |  V5 → V8 → V9 → QML-clean → QML-noisy  |  normalised eval
  Endpoint: https://integrate.api.nvidia.com/v1
  Model   : meta/llama-3.1-8b-instruct
  Rails   : NeMo Guardrails (Colang 1.0) — output checks active

[1/3] Scientist agent...
  ✓  latency=9257.8ms  TTFT=1665.8ms  tokens/s=46.6  guard=✅
[2/3] Critic agent...
  ✓  latency=19307.5ms  TTFT=14085.0ms  tokens/s=25.1  guard=✅
[3/3] QML Synthesiser agent...
  ✓  latency=49869.2ms  TTFT=44715.5ms  tokens/s=8.0  guard=✅

── Pipeline complete ────────────────────────────────────────────
  Total pipeline latency : 78434ms
  Guardrail pass rate    : 100%
  Agents fired           : 3  (scientist / critic / qml_synthesiser)
  Rails                  : NeMo Guardrails

── Guardrail audit ──────────────────────────────────────────────
  scientist          ✅ PASS    

### Cell 17 — Inference benchmark harness

In [36]:
# Cell 17 — Inference benchmark (run N times, collect latency + guardrail stats)
# Set N_BENCHMARK_RUNS = 1 to skip re-running (uses results from cell 16)
N_BENCHMARK_RUNS = 5   # increase to 5 or 10 for publication-quality numbers

if N_BENCHMARK_RUNS <= 1:
    # Use the single run from cell 16 directly
    bench_records = results_agents["benchmark_records"]
    print("Using single-run benchmark from cell 16.")
    print("Set N_BENCHMARK_RUNS > 1 to run a proper benchmark sweep.")
else:
    # Run multiple times and aggregate
    from benchmark.nim_benchmark import NIMBenchmark, default_guardrail_check
    bm = NIMBenchmark(
        endpoint=nag.NIM_BASE_URL,
        api_key=nag.NVIDIA_API_KEY,
        model=nag.NIM_MODEL,
    )

    # Build a compact payload for benchmark runs (no full attention data)
    lm = agent_stats["live_metrics"]
    bench_payload = json.dumps({"live_metrics": lm}, indent=2)

    bm_report = await bm.run_benchmark(
        scientist_system=nag.SCIENTIST_SYSTEM,
        critic_system=nag.CRITIC_SYSTEM,
        qml_system=nag.QML_SYSTEM,
        user_payload=bench_payload,
        n_runs=N_BENCHMARK_RUNS,
        guardrail_check_fn=default_guardrail_check,
    )
    bench_records = [c for calls in bm_report.agent_reports.values() for c in calls]

# ── Display benchmark table ───────────────────────────────────────────────────
import pandas as pd
df_bench = pd.DataFrame([
    {
        "agent":            r["agent"] if isinstance(r, dict) else r.agent,
        "ttft_ms":          r["ttft_ms"] if isinstance(r, dict) else r.ttft_ms,
        "latency_ms":       r["total_ms"] if isinstance(r, dict) else r.total_ms,
        "tokens_per_sec":   r["tokens_per_sec"] if isinstance(r, dict) else r.tokens_per_sec,
        "guardrail_pass":   r["guardrail_pass"] if isinstance(r, dict) else r.guardrail_pass,
        "guardrail_fired":  r["guardrail_fired"] if isinstance(r, dict) else r.guardrail_fired,
    }
    for r in bench_records
])
display(df_bench.to_string(index=False))
print(f"\nModel scores (from agent_stats):")
print(f"  V9  BLEU-1={agent_stats['live_metrics']['v9_tf_bleu1_pct']}%  "
      f"ROUGE-1={agent_stats['live_metrics']['v9_tf_rouge1_pct']}%")
print(f"  QML BLEU-1={agent_stats['live_metrics']['qml_tf_bleu1_pct']}%  "
      f"ROUGE-1={agent_stats['live_metrics']['qml_tf_rouge1_pct']}%")



  Benchmarking 5 pipeline runs on https://integrate.api.nvidia.com/v1...
  Run 1/5... done
  Run 2/5... done
  Run 3/5... done
  Run 4/5... done
  Run 5/5... done

  INFERENCE BENCHMARK REPORT
  Endpoint : https://integrate.api.nvidia.com/v1
  Model    : meta/llama-3.1-8b-instruct

── SCIENTIST ─────────────────────────────────────
  Runs: 5  Success: 5  Guardrail pass: 40.0%
  TTFT    mean=5611.9ms  p95=8009.4ms
  Latency mean=15535.1ms  p95=17176.4ms
  Tokens/s mean=80.5  Output tokens mean=1224.8

── CRITIC ─────────────────────────────────────
  Runs: 5  Success: 5  Guardrail pass: 100.0%
  TTFT    mean=5828.4ms  p95=9188.1ms
  Latency mean=11409.7ms  p95=16159.2ms
  Tokens/s mean=68.0  Output tokens mean=660.2

── QML_SYNTHESISER ─────────────────────────────────────
  Runs: 5  Success: 5  Guardrail pass: 100.0%
  TTFT    mean=3255.1ms  p95=6001.7ms
  Latency mean=7860.6ms  p95=12272.1ms
  Tokens/s mean=75.4  Output tokens mean=459.4

── FULL PIPELINE (3 agents sequential: scient

'          agent  ttft_ms  latency_ms  tokens_per_sec  guardrail_pass                  guardrail_fired\n      scientist   7883.8     14774.7            66.9           False metric_out_of_range:bleu-1=6.19%\n      scientist    858.3     13515.0            96.7           False  metric_out_of_range:bleu-1=6.9%\n      scientist   2102.5     11834.3            92.9           False  metric_out_of_range:bleu1=0.57%\n      scientist   8009.4     17176.4            76.9            True                                 \n      scientist   9205.5     20375.1            69.1            True                                 \n         critic   9188.1     16159.2            45.5            True                                 \n         critic   5034.0     10574.9            64.5            True                                 \n         critic  11885.6     17674.6            37.9            True                                 \n         critic   2538.6      6295.7            85.8            True    


Model scores (from agent_stats):
  V9  BLEU-1=30.97%  ROUGE-1=36.07%
  QML BLEU-1=30.95%  ROUGE-1=36.04%


### Cell 18 — Display agent outputs

In [37]:
# Cell 18 — Display agent outputs
from IPython.display import Markdown, display

# ── Diagnostic: check what came back before rendering ────────────────────────
def _show_agent(label, key):
    text = results_agents.get(key, "")
    if not text:
        print(f"⚠  {label}: empty string — cell 16 may not have run successfully")
        print(f"   Re-run cell 16 to regenerate agent outputs.")
        display(Markdown(f"---\n## {label}\n*No output — re-run cell 16*"))
        return
    if text.startswith("[error:") or text.startswith("[simulation"):
        print(f"⚠  {label}: error/simulation response — {text[:100]}")
        display(Markdown(f"---\n## {label}\n```\n{text[:300]}\n```"))
        return
    display(Markdown(f"---\n## {label}"))
    display(Markdown(text))

_show_agent("Scientist Agent",  "scientist")
_show_agent("Critic Agent",     "critic")
_show_agent("QML Synthesiser",  "qml_synthesiser")

# ── Metric trace table ────────────────────────────────────────────────────────
lm   = results_agents["stats"]["live_metrics"]
v5   = results_agents["stats"]["baselines"]["v5"]
v8   = results_agents["stats"]["baselines"]["v8"]
aa   = results_agents["stats"]["attention_analysis"]
c_aa = aa["v9_classical"]["cross_region_fusion"]
h_aa = aa["v9_qml_hybrid"]["cross_region_fusion"]

display(Markdown(f"""---
## Agent trace — V5 / V8 / V9 / V9+QML  (val n={lm["n"]:,})

| Metric | V5 | V8 | V9 classical | V9+QML hybrid |
|--------|----|----|--------------|---------------|
| TF BLEU-1 | {v5["tf_bleu1_pct"]}% | {v8["tf_bleu1_pct"]}% | **{lm["v9_tf_bleu1_pct"]}%** | **{lm["qml_tf_bleu1_pct"]}%** |
| TF BLEU-4 | — | {v8["tf_bleu4_pct"]}% | {lm["v9_tf_bleu4_pct"]}% | {lm["qml_tf_bleu4_pct"]}% |
| TF ROUGE-1 | {v5["tf_rouge1_pct"]}% | {v8["tf_rouge1_pct"]}% | {lm["v9_tf_rouge1_pct"]}% | {lm["qml_tf_rouge1_pct"]}% |
| TF ROUGE-L | {v5["tf_rougeL_pct"]}% | {v8["tf_rougeL_pct"]}% | {lm["v9_tf_rougeL_pct"]}% | {lm["qml_tf_rougeL_pct"]}% |
| FG BLEU-1 | — | {v8["fg_bleu1_pct"]}% | {lm["v9_fg_bleu1_pct"]}% | {lm["qml_fg_bleu1_pct"]}% |
| TF/FG ratio | — | {v8["tf_fg_ratio"]}x | {lm["v9_tf_fg_ratio"]}x | {lm["qml_tf_fg_ratio"]}x |
| BERTScore F1 | — | {v8["bertscore_f1"]}% | — | — |

| Delta pair | BLEU-1 | ROUGE-1 |
|-----------|--------|---------|
| V5 → V8 | +1.16pp | +2.09pp |
| V8 → V9 | {lm["delta_v9_vs_v8_bleu1"]:+.2f}pp | {lm["delta_v9_vs_v8_rouge1"]:+.2f}pp |
| V9 → QML | {lm["delta_qml_vs_v9_bleu1"]:+.2f}pp | {lm["delta_qml_vs_v9_rouge1"]:+.2f}pp |
| V8 → QML | {lm["delta_qml_vs_v8_bleu1"]:+.2f}pp | — |

**Per-condition TF BLEU-1:**

| Condition | V5 | V8 | V9 | QML | Δ V9–V8 | Δ QML–V9 |
|-----------|----|----|----|----|---------|----------|
| NR  | {v5["per_condition"]["NR"]}%  | {v8["per_condition"]["NR"]}%  | {lm["v9_per_cond_bleu1"].get("NR","?")}%  | {lm["qml_per_cond_bleu1"].get("NR","?")}%  | {round(lm["v9_per_cond_bleu1"].get("NR",0)  - v8["per_condition"]["NR"],  2):+.2f}pp | {round(lm["qml_per_cond_bleu1"].get("NR",0)  - lm["v9_per_cond_bleu1"].get("NR",0),  2):+.2f}pp |
| TSR | {v5["per_condition"]["TSR"]}% | {v8["per_condition"]["TSR"]}% | {lm["v9_per_cond_bleu1"].get("TSR","?")}% | {lm["qml_per_cond_bleu1"].get("TSR","?")}% | {round(lm["v9_per_cond_bleu1"].get("TSR",0) - v8["per_condition"]["TSR"], 2):+.2f}pp | {round(lm["qml_per_cond_bleu1"].get("TSR",0) - lm["v9_per_cond_bleu1"].get("TSR",0), 2):+.2f}pp |
| SR  | {v5["per_condition"]["SR"]}%  | {v8["per_condition"]["SR"]}%  | {lm["v9_per_cond_bleu1"].get("SR","?")}%  | {lm["qml_per_cond_bleu1"].get("SR","?")}%  | {round(lm["v9_per_cond_bleu1"].get("SR",0)  - v8["per_condition"]["SR"],  2):+.2f}pp | {round(lm["qml_per_cond_bleu1"].get("SR",0)  - lm["v9_per_cond_bleu1"].get("SR",0),  2):+.2f}pp |
"""))

# ── Attention table (pre-built rows — avoids TypeError) ───────────────────────
_attn_rows = []
for _name in REGION_NAMES:
    _v9  = c_aa["values"].get(_name, 0)
    _qml = h_aa["values"].get(_name, 0)
    _nr  = (c_aa["per_condition"].get("NR")  or {}).get(_name, 0)
    _tsr = (c_aa["per_condition"].get("TSR") or {}).get(_name, 0)
    _sr  = (c_aa["per_condition"].get("SR")  or {}).get(_name, 0)
    _attn_rows.append(f"| {_name} | {_v9:.4f} | {_qml:.4f} | {_nr:.4f} | {_tsr:.4f} | {_sr:.4f} |")
_attn_table = "\n".join(_attn_rows)

display(Markdown(f"""---
## Attention diagnostic — cross-region fusion (self.fusion MHA)

| Region | V9 overall | QML overall | NR | TSR | SR |
|--------|-----------|-------------|-----|-----|-----|
{_attn_table}

**V9 dominant:** {c_aa["dominant"]}  
**QML dominant:** {h_aa["dominant"]}
"""))

# ── Pipeline benchmark ────────────────────────────────────────────────────────
ps = results_agents.get("pipeline_summary", {})
if ps:
    print(f"\n── Pipeline summary ─────────────────────────────────────────────")
    print(f"  Total latency  : {ps.get('total_pipeline_ms', '?')}ms")
    print(f"  Guardrail pass : {ps.get('guardrail_pass_rate_pct', '?')}%")
    print(f"  Rails active   : {ps.get('rails_active', False)}")
    print(f"  Endpoint       : {ps.get('endpoint', '?')}")


---
## Scientist Agent

**1. DATASET & SETUP**

The analysis is based on the ZuCo dataset, which is a collection of EEG recordings from individuals performing various cognitive tasks. The dataset is split into training and validation sets, with the validation set used for evaluating the performance of the models. The conditions in the dataset include Normal Reading (NR), Transient Sentence Reading (TSR), and Sentence Reading (SR). The setup involves normalizing the EEG, eye, and spectral data using saved scalers.

**2. FOUR-MODEL PROGRESSION**

The four models analyzed are V5, V8, V9, and QML. Each model represents an architectural addition to the previous one. The progression is as follows:

- V5: This is the initial model, which uses a Conv1D + Bi-GRU + single mean-pool EEG vector and a prefix-tuned DistilGPT2.
- V8: This model adds 6 parallel GRU-Transformer RegionEncoders, MoCo Stage0, LoRA rank=8 GPT2, and SR adapter.
- V9: This model adds HierarchicalTemporalPooling (HTP) to the V8 architecture, which includes local_attn and seg_attn per region.
- QML: This model adds QuantumFusionProjector to the V9 architecture, which includes down H→4, AngleEmbedding, 2 StronglyEntanglingLayers, and up 4→H.

The justification for each architectural addition is based on the performance metrics. The addition of HTP in V9 led to a significant improvement in ROUGE-1 and ROUGE-L scores. The addition of QuantumFusionProjector in QML led to a slight improvement in ROUGE-1 and ROUGE-L scores.

**3. TF PERFORMANCE**

The performance of the four models in terms of TF metrics is as follows:

- V5: TF BLEU-1=29.24%, ROUGE-1=33.92%, ROUGE-L=30.06%
- V8: TF BLEU-1=30.4%, ROUGE-1=35.78%, ROUGE-L=30.68%, BERTScore=85.46%, FG=4.91%
- V9: TF BLEU-1=30.97%, ROUGE-1=36.07%, ROUGE-L=30.76%, FG=6.9%, TF/FG=4.49x
- QML: TF BLEU-1=30.95%, ROUGE-1=36.04%, ROUGE-L=30.77%, FG=6.88%, TF/FG=4.5x
- NQM: TF BLEU-1=30.95%, ROUGE-1=36.05%, ROUGE-L=30.76%, FG=6.81%

The performance of the models improves with each addition, with V9 and QML showing the best results.

**4. FG PERFORMANCE**

The performance of the four models in terms of FG metrics is as follows:

- V8: FG=4.91%
- V9: FG=6.9%
- QML: FG=6.88%
- NQM: FG=6.81%

The addition of HTP in V9 led to a significant improvement in FG score, and the addition of QuantumFusionProjector in QML led to a slight improvement.

**5. PER-CONDITION**

The performance of the four models in terms of per-condition TF BLEU-1 is as follows:

- V5: NR=30.7%, TSR=32.78%, SR=26.49%
- V8: NR=30.9%, TSR=32.93%, SR=27.2%
- V9: NR=31.53%, TSR=33.77%, SR=27.46%
- QML: NR=31.43%, TSR=33.87%, SR=27.37%
- NQM: NR=31.44%, TSR=33.93%, SR=27.3%

The performance of the models improves with each addition, with V9 and QML showing the best results.

**6. ATTENTION DIAGNOSIS**

a) **HTP:** The addition of HTP in V9 led to a significant improvement in ROUGE-1 and ROUGE-L scores

---
## Critic Agent

[ISSUE-1] HTP: genuine improvement vs parameter count increase?
Problem: The Hierarchical Temporal Pooling (HTP) module introduces local_attn + seg_attn, replacing collapsed pool_attn, but the impact on performance and parameter efficiency is unclear.
Fix: Provide a detailed analysis of the HTP's contribution to the model's performance, including a comparison of the number of parameters added and the resulting improvement in metrics.

[ISSUE-2] QML clean: expressivity advantage vs equivalent 4→768 classical MLP residual?
Problem: The Quantum Fusion Projector (QML) clean model shows a slight improvement over V9, but its expressivity advantage over an equivalent classical MLP residual is not clear.
Fix: Compare the QML clean model's performance with an equivalent classical MLP residual, and provide a detailed analysis of the QML's expressivity advantage.

[ISSUE-3] QML noisy: val loss 4.1729 vs clean 4.1733 — is 0.0004 gap meaningful at this scale?
Problem: The QML noisy model shows a slight decrease in validation loss compared to the clean model, but the gap of 0.0004 is not significant at this scale.
Fix: Provide a more detailed analysis of the QML noisy model's performance, including a comparison of the validation loss and its standard deviation.

[ISSUE-4] Statistical significance of QML delta at ZuCo scale (~2032 val samples)
Problem: The QML model shows a slight improvement over V9, but the statistical significance of this delta at the ZuCo scale is unclear.
Fix: Provide a statistical analysis of the QML model's performance, including a comparison of the p-value and effect size.

[ISSUE-5] TF/FG ratio progress vs V8 6.19× baseline means same lower the ratio better the model
Problem: The TF/FG ratio has decreased from V8 to V9 and QML, but the interpretation of this trend is unclear.
Fix: Provide a detailed analysis of the TF/FG ratio's trend and its implications for the model's performance.

[ISSUE-6] eval protocol comparability (same split? same normalisation? same logit shift?)
Problem: The evaluation protocol used for the QML model is unclear, and its comparability to the previous models is not established.
Fix: Provide a detailed description of the evaluation protocol used for the QML model, including the split, normalization, and logit shift.

[ISSUE-7] LoRA rank reduction from 8→4: ablation missing?
Problem: The LoRA rank reduction from 8 to 4 is not accompanied by an ablation study to understand its impact on the model's performance.
Fix: Provide an ablation study to understand the impact of the LoRA rank reduction on the model's performance.

Correctly identified:
* The Hierarchical Temporal Pooling (HTP) module introduces local_attn + seg_attn, replacing collapsed pool_attn.
* The Quantum Fusion Projector (QML) clean model shows a slight improvement over V9.
* The QML noisy model shows a slight decrease in validation loss compared to the clean model.
* The TF/FG ratio has decreased from V8 to V9 and QML.

Verdict: CONDITIONAL PASS
Confidence: 6/10 — The submission provides some interesting results, but the analysis is not thorough enough to establish the significance of the findings.

---
## QML Synthesiser

The Hierarchical Temporal Pooling (HTP) module in the V8 and V9 models introduces a significant increase in parameter count, which raises questions about its impact on performance. In the V8 model, the addition of local_attn and seg_attn modules replaces the collapsed pool_attn, but the effect on the model's performance is unclear. A detailed analysis of the HTP's contribution to the model's performance is necessary to determine whether the increase in parameter count is justified by the improvement in metrics.

The QML model, which is based on the QuantumFusionProjector (QFP) architecture, exhibits a similar performance to the V9 model, with a validation loss of 4.1729. However, the QFP clean variant, which is trained on a noiseless statevector simulation, has a slightly higher validation loss of 4.1733. The difference in validation loss between the QFP clean and QFP noisy variants is only 0.0004, which raises questions about the role of noise as a regulariser in the QFP architecture. This small difference in validation loss may be due to the noise-aware fine-tuning of the QFP noisy variant, which could be masking the true expressivity advantage of the QFP architecture.

The QFP architecture itself is based on a 4-qubit, 2-layer Variational Quantum Circuit (VQC), which is a significant departure from the classical equivalent baseline. The VQC is composed of a series of strongly entangling layers, which are followed by a linear layer and a layer norm. The QFP architecture also includes a residual connection, which allows the model to learn a more complex representation of the input data. However, the true advantage of the QFP architecture is unclear, and further analysis is necessary to determine whether the small difference in validation loss is due to the quantum computation or simply the noise-aware fine-tuning.

The progression from V5 to V8 to V9 to QML provides valuable insights into the development of EEG-to-text models. The introduction of HTP in the V8 and V9 models improves performance, but at the cost of increased parameter count. The QML model, which is based on the QFP architecture, exhibits similar performance to the V9 model, but with a more complex and expressive architecture. The next step in this progression is to further investigate the QFP architecture and its potential advantages over classical models. This could involve exploring different variants of the QFP architecture, such as increasing the number of qubits or layers, or introducing different types of noise or regularisation.

---
## Agent trace — V5 / V8 / V9 / V9+QML  (val n=2,032)

| Metric | V5 | V8 | V9 classical | V9+QML hybrid |
|--------|----|----|--------------|---------------|
| TF BLEU-1 | 29.24% | 30.4% | **30.97%** | **30.95%** |
| TF BLEU-4 | — | 4.3% | 4.45% | 4.46% |
| TF ROUGE-1 | 33.92% | 36.01% | 36.07% | 36.04% |
| TF ROUGE-L | 30.06% | 30.9% | 30.76% | 30.77% |
| FG BLEU-1 | — | 4.92% | 6.9% | 6.88% |
| TF/FG ratio | — | 6.19x | 4.49x | 4.5x |
| BERTScore F1 | — | 85.46% | — | — |

| Delta pair | BLEU-1 | ROUGE-1 |
|-----------|--------|---------|
| V5 → V8 | +1.16pp | +2.09pp |
| V8 → V9 | +0.57pp | +0.06pp |
| V9 → QML | -0.02pp | -0.03pp |
| V8 → QML | +0.55pp | — |

**Per-condition TF BLEU-1:**

| Condition | V5 | V8 | V9 | QML | Δ V9–V8 | Δ QML–V9 |
|-----------|----|----|----|----|---------|----------|
| NR  | 30.7%  | 30.9%  | 31.53%  | 31.43%  | +0.63pp | -0.10pp |
| TSR | 32.78% | 32.93% | 33.77% | 33.87% | +0.84pp | +0.10pp |
| SR  | 26.49%  | 27.2%  | 27.46%  | 27.37%  | +0.26pp | -0.09pp |


---
## Attention diagnostic — cross-region fusion (self.fusion MHA)

| Region | V9 overall | QML overall | NR | TSR | SR |
|--------|-----------|-------------|-----|-----|-----|
| left_temporal | 0.1821 | 0.1821 | 0.1825 | 0.1797 | 0.1843 |
| left_parietal | 0.2283 | 0.2283 | 0.2293 | 0.2259 | 0.2298 |
| left_parieto_occipital | 0.2519 | 0.2519 | 0.2405 | 0.2691 | 0.2445 |
| central_parietal | 0.1084 | 0.1084 | 0.1127 | 0.1036 | 0.1095 |
| right_parietal | 0.1132 | 0.1132 | 0.1148 | 0.1110 | 0.1143 |
| right_parieto_occipital | 0.1161 | 0.1161 | 0.1203 | 0.1108 | 0.1177 |

**V9 dominant:** left_parieto_occipital  
**QML dominant:** left_parieto_occipital



── Pipeline summary ─────────────────────────────────────────────
  Total latency  : 78434.5ms
  Guardrail pass : 100.0%
  Rails active   : True
  Endpoint       : https://integrate.api.nvidia.com/v1


### Cell 19 — Save results

In [22]:
# Cell 19 — Save results (now includes benchmark records + guardrail audit)
out = BASE / "nat_v9_qml_results.json"
with open(out, "w") as f:
    json.dump({
        "stats":             json.loads(json.dumps(results_agents["stats"], default=str)),
        "scientist":         results_agents["scientist"],
        "critic":            results_agents["critic"],
        "qml_synthesiser":   results_agents["qml_synthesiser"],
        "explainer":         results_agents["explainer"],   # backward-compat
        "benchmark_records": results_agents["benchmark_records"],
        "guardrail_audit":   results_agents["guardrail_audit"],
        "pipeline_summary":  results_agents["pipeline_summary"],
    }, f, indent=2)

lm  = results_agents["stats"]["live_metrics"]
ps  = results_agents["pipeline_summary"]
print(f"✓ Saved → {out}")
print(f"  V5  TF BLEU-1 : {V5_BASELINE['tf_bleu1_pct']}%")
print(f"  V8  TF BLEU-1 : {V8_BASELINE['tf_bleu1_pct']}%")
print(f"  V9  TF BLEU-1 : {lm['v9_tf_bleu1_pct']}%  (Δ vs V8: {lm['delta_v9_vs_v8_bleu1']:+.2f}pp)")
print(f"  QML TF BLEU-1 : {lm['qml_tf_bleu1_pct']}%  (Δ vs V9: {lm['delta_qml_vs_v9_bleu1']:+.2f}pp)")
print(f"  Pipeline latency: {ps['total_pipeline_ms']}ms")
print(f"  Guardrail pass  : {ps['guardrail_pass_rate_pct']}%")
print(f"  Endpoint        : {ps['endpoint']}")


✓ Saved → nat_v9_qml_results.json
  V5  TF BLEU-1 : 29.24%
  V8  TF BLEU-1 : 30.4%
  V9  TF BLEU-1 : 30.97%  (Δ vs V8: +0.57pp)
  QML TF BLEU-1 : 30.95%  (Δ vs V9: -0.02pp)
  Pipeline latency: 176497.0ms
  Guardrail pass  : 100.0%
  Endpoint        : https://integrate.api.nvidia.com/v1
